# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamnahFatimaKhan/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **Grain / Unit of Analysis:** One row = One published URL (`page_path`) observed over a monthly aggregation window.
2. **Tables Used:** `content_refresh_anonymized.csv` (representing warehouse landing tables for search console and traffic performance).
3. **Time Window:** Mid-panel observation period (March 2026 / `month=2026-03`).
4. **Target / Proxy:** `is_declining_label` (Binary indicator: 1 if 90-day performance trend is down, 0 otherwise).
5. **Deliberately Excluded:** Post-decision future metrics (e.g., next month's clicks/impressions or post-update traffic gains) to prevent lookahead bias.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import subprocess
import pandas as pd
import numpy as np

# Ensure repo is cloned and working directory is correct
REPO_NAME = "Flyrank-ML-Internship"
REPO_URL = f"https://github.com/HamnahFatimaKhan/{REPO_NAME}.git"

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if os.path.exists(f"../../data/raw/content_refresh_anonymized.csv"):
        os.chdir("../..")
    elif not os.path.exists(REPO_NAME):
        subprocess.run(["git", "clone", REPO_URL], check=True)
        os.chdir(REPO_NAME)
    else:
        os.chdir(REPO_NAME)

# Load starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target label
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Identify unique path column dynamically
path_col = [col for col in ["page_path", "content_path", "url_path", "path"] if col in df.columns]
path_col = path_col[0] if path_col else df.columns[0]

print("--- Query 1: Verify Grain ---")
total_rows = len(df)
unique_pages = df[path_col].nunique()
print(f"Total rows: {total_rows} | Unique items ({path_col}): {unique_pages} (1 row = 1 unit)")

print("\n--- Query 2: Row Count & Date Span ---")
print(f"Total Rows in Slice: {len(df)}")
print(f"Max Content Age Days: {df['content_age_days'].max()} | Min Days Since Update: {df['days_since_last_update'].min()}")

print("\n--- Query 3: Availability Filtering (IS TRUE) ---")
surviving_rows = df[df["impressions_90d"] >= 10]
print(f"Rows surviving 'impressions_90d >= 10 IS TRUE': {len(surviving_rows)} / {len(df)} ({len(surviving_rows)/len(df):.1%})")

--- Query 1: Verify Grain ---
Total rows: 30000 | Unique items (content_id): 30000 (1 row = 1 unit)

--- Query 2: Row Count & Date Span ---
Total Rows in Slice: 30000
Max Content Age Days: 564 | Min Days Since Update: 1

--- Query 3: Availability Filtering (IS TRUE) ---
Rows surviving 'impressions_90d >= 10 IS TRUE': 26254 / 30000 (87.5%)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Build feature set
features = ["days_since_last_update", "content_age_days", "impressions_90d", "avg_position", "ctr"]
X = df[features].copy()
y = df["is_declining_label"]

# --- THE TRAP: Adding a Leaked Column ---
X["LEAKED_trend_score"] = df["is_declining_label"] * 0.99  # Direct lookahead leakage

# Fit model with leaked features
model_leaked = DecisionTreeClassifier().fit(X, y)
acc_leaked = accuracy_score(y, model_leaked.predict(X))
print(f"Accuracy WITH Leakage: {acc_leaked:.1%}")

# --- REMOVING THE LEAK ---
X_honest = X.drop(columns=["LEAKED_trend_score"])
model_honest = DecisionTreeClassifier(max_depth=3).fit(X_honest, y)
acc_honest = accuracy_score(y, model_honest.predict(X_honest))
print(f"Accuracy WITHOUT Leakage (Honest Baseline): {acc_honest:.1%}")

Accuracy WITH Leakage: 100.0%
Accuracy WITHOUT Leakage (Honest Baseline): 64.5%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Limitation:** The mid-panel snapshot reflects static performance over past aggregations. It cannot capture real-time Search Engine Algorithm updates (e.g., Google Core Updates) or sudden competitor content releases that occur immediately after the decision window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.